In [1]:
import os
import s3fs
import pandas as pd

os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"

fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]},
    skip_instance_cache=True,
)

pd.set_option("display.max_columns", None)

In [2]:
chemin_bpe = "samuelbarre/diffusion/bpe_alim_2025.parquet"
chemin_dvf = "samuelbarre/diffusion/dvf_2025.parquet"

with fs.open(chemin_bpe, "rb") as f:
    bpe = pd.read_parquet(f)

with fs.open(chemin_dvf, "rb") as f:
    dvf = pd.read_parquet(f)

print("BPE :", bpe.shape)
print("DVF :", dvf.shape)

BPE : (135587, 21)
DVF : (930120, 14)


In [4]:
codes_b2 = sorted(c for c in bpe["TYPEQU"].unique() if c.startswith("B2"))

types_commerces = {
    "hyper": ["B104"],
    "super": ["B105"],
    "proximite": ["B201", "B202"],
    "specialise": [c for c in codes_b2 if c not in ["B201", "B202"]],
}

for nom, codes in types_commerces.items():
    print(nom, ":", codes, "->", bpe["TYPEQU"].isin(codes).sum(), "commerces")


hyper : ['B104'] -> 2081 commerces
super : ['B105'] -> 12106 commerces
proximite : ['B201', 'B202'] -> 30575 commerces
specialise : ['B204', 'B205', 'B206', 'B207', 'B208', 'B209', 'B210'] -> 90825 commerces


In [11]:
for nom, codes in types_commerces.items():
    commerces = bpe.loc[bpe["TYPEQU"].isin(codes), ["geometry"]]

    plus_proche_type = gpd.sjoin_nearest(
        dvf[["geometry"]], commerces,
        how="left", distance_col=f"dist_{nom}",
    )
    plus_proche_type = plus_proche_type[~plus_proche_type.index.duplicated()]

    dvf[f"dist_{nom}"] = plus_proche_type[f"dist_{nom}"]
    print(f"{nom} : distance médiane {dvf[f'dist_{nom}'].median():.0f} m")

hyper : distance médiane 2309 m
super : distance médiane 748 m
proximite : distance médiane 476 m
specialise : distance médiane 284 m


In [12]:
commerces_alim = bpe.loc[
    bpe["TYPEQU"].isin(sum(types_commerces.values(), [])),
    ["TYPEQU", "geometry"],
]

plus_proche = gpd.sjoin_nearest(
    dvf[["geometry"]], commerces_alim,
    how="left", distance_col="dist_plus_proche",
)
plus_proche = plus_proche[~plus_proche.index.duplicated()]

noms_types = {code: nom for nom, codes in types_commerces.items() for code in codes}
dvf["dist_plus_proche"] = plus_proche["dist_plus_proche"]
dvf["type_plus_proche"] = plus_proche["TYPEQU"].map(noms_types)

dvf["type_plus_proche"].value_counts(normalize=True).round(3)

type_plus_proche
specialise    0.650
proximite     0.240
super         0.099
hyper         0.012
Name: proportion, dtype: float64

In [13]:
colonnes_dist = ["dist_hyper", "dist_super", "dist_proximite", "dist_plus_proche"]
dvf[colonnes_dist].describe(percentiles=[0.1, 0.5, 0.9]).round(0)

,dist_hyper,dist_super,dist_proximite,dist_plus_proche
count,930120.0,930120.0,930120.0,930120.0
mean,5080.0,1940.0,1320.0,627.0
std,7407.0,5002.0,4652.0,4355.0
min,0.0,0.0,0.0,0.0
10%,646.0,168.0,81.0,43.0
50%,2309.0,748.0,476.0,237.0
90%,13515.0,5770.0,4014.0,1815.0
max,4082630.0,4076927.0,4075887.0,4075887.0


In [14]:
densite7_communes = bpe.groupby("DEPCOM")["DENS7"].first()
dvf["DENS7"] = dvf["code_commune"].map(densite7_communes)

print(dvf["DENS7"].value_counts(dropna=False).sort_index())

DENS7
1      340080
2      136388
3       62959
4       75605
5      136390
6       90801
7       12403
NaN     75494
Name: count, dtype: int64


In [19]:
codes_bpe = set(bpe["DEPCOM"].dropna().astype(str))
codes_dvf = dvf["code_commune"].astype(str)

print(
    "Part des transactions DVF dont la commune est dans la BPE :",
    round(codes_dvf.isin(codes_bpe).mean() * 100, 2), "%"
)

print(
    "Nombre de communes différentes dans la BPE :",
    len(codes_bpe)
)

Part des transactions DVF dont la commune est dans la BPE : 91.88 %
Nombre de communes différentes dans la BPE : 15300
